# 4. Популярность ноутбуков
Первый взгляд на голоса и просмотры, связь с тегами и годом публикации.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load

sns.set_theme(style="whitegrid")
kernels = load("Kernels")
kernels = kernels[kernels["MadePublicDate"].notna()].copy()  # только публичные
kernels[["TotalViews", "TotalVotes", "TotalComments"]].describe()

Распределения сильно скошены — смотрим в логарифме.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
np.log1p(kernels["TotalVotes"]).plot(kind="hist", bins=50, ax=ax[0], title="log(1 + голоса)")
np.log1p(kernels["TotalViews"]).plot(kind="hist", bins=50, ax=ax[1], title="log(1 + просмотры)")
plt.savefig("../reports/figures/04_votes_views_hist.png", bbox_inches="tight")
plt.show()

In [ ]:
kernels[["TotalViews", "TotalVotes", "TotalComments"]].corr(method="spearman")

## Год публикации

In [ ]:
kernels["year"] = kernels["MadePublicDate"].dt.year
kernels.groupby("year")["TotalVotes"].agg(["count", "median", "mean"])

## Теги

In [ ]:
tags = load("Tags")
kt = load("KernelTags").merge(tags, left_on="TagId", right_on="Id")
kt = kt.merge(kernels[["Id", "TotalVotes"]], left_on="KernelId", right_on="Id", suffixes=("", "_k"))

tag_stats = kt.groupby("Name")["TotalVotes"].agg(["count", "median"])
tag_stats[tag_stats["count"] >= 500].sort_values("median", ascending=False).head(20)

TODO: добавить датасеты (Datasets), признаки автора, кластеризация тегов.